### Imports and Load

In [1]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.model_selection import train_test_split

# Paths
PROCESSED_PATH = Path.cwd().parent / "data" / "processed"
OUTPUT_PATH    = Path.cwd().parent / "data" / "model" / "scaled"

# Column config
TARGET_COL    = "label"
METADATA_COLS = ["scenario_id", "timestep"]
NON_ML_COLS   = ["fault_type", "effect_factor"]

# Split config
RANDOM_SEED = 42

# Scalers to compare
SCALERS = {
    "robust":   RobustScaler(),
    "standard": StandardScaler(),
}

In [2]:
def load_data(path: Path, filename: str) -> pd.DataFrame:
    df = pd.read_csv(path / filename)
    print(f"Loaded shape : {df.shape}")
    print(f"Label distribution:\n{df[TARGET_COL].value_counts().to_string()}\n")
    return df

df = load_data(PROCESSED_PATH, "engineered_dataset.csv")
df.head()

Loaded shape : (31500, 131)
Label distribution:
label
2    10500
1    10500
0    10500



,scenario_id,timestep,node_a_pressure,velocity_a,tke_a,tdr_a,wall_shear_a,tailings_vof_a,node_b_pressure,velocity_b,...,rolling_mean_velocity_b,rolling_std_velocity_b,rolling_mean_pressure_drop_bc,rolling_std_pressure_drop_bc,rolling_mean_midpoint_pressure_deviation,rolling_std_midpoint_pressure_deviation,rolling_mean_midpoint_velocity_deviation,rolling_std_midpoint_velocity_deviation,rolling_mean_midpoint_vof_deviation,rolling_std_midpoint_vof_deviation
0,blockage_25_loc16m,1,35793.499725,2.534141,0.034635,24.489446,11.197253,0.401012,19965.653843,2.474028,...,2.474028,0.000000,15575.231918,0.000000,-126.306983,0.000000,-0.056303,0.000000,-0.001483,0.000000
1,blockage_25_loc16m,2,36620.208522,2.484805,0.038275,25.658411,10.641124,0.405089,20589.416498,2.421521,...,2.447775,0.037128,15784.963909,296.605826,-72.177522,76.550617,-0.066276,0.014104,-0.002528,0.001478
2,blockage_25_loc16m,3,36251.794105,2.473708,0.034279,25.294662,9.484428,0.398871,20946.810911,2.531422,...,2.475657,0.054968,16232.488105,803.007614,255.640536,570.371841,-0.013167,0.092527,0.000671,0.005639
3,blockage_25_loc16m,4,35694.151655,2.478258,0.032199,27.022206,9.489620,0.393614,19637.008191,2.406712,...,2.458421,0.056592,16108.980877,700.640394,151.894868,509.838549,-0.032113,0.084518,0.002760,0.006216
4,blockage_25_loc16m,5,35834.383052,2.467021,0.037261,22.894842,10.660725,0.400522,19776.413014,2.442424,...,2.455221,0.049530,16234.319894,668.372701,189.286487,449.379767,-0.035610,0.073611,0.001922,0.005700


### Define Columns

In [3]:
def get_feature_columns(df: pd.DataFrame) -> list[str]:
    exclude = set(METADATA_COLS + NON_ML_COLS + [TARGET_COL])
    feature_cols = [c for c in df.columns if c not in exclude]
    print(f"Total ML features : {len(feature_cols)}")
    print(f"Excluded          : {sorted(exclude)}")
    return feature_cols

feature_cols = get_feature_columns(df)

Total ML features : 126
Excluded          : ['effect_factor', 'fault_type', 'label', 'scenario_id', 'timestep']


### Scenario Level Split


In [4]:
def split_scenario_ids(scenarios, seed=RANDOM_SEED):
    """75 / 12.5 / 12.5 split on scenario IDs — NOT on rows."""
    train, temp = train_test_split(scenarios, test_size=0.25, random_state=seed)
    val, test   = train_test_split(temp,      test_size=0.50, random_state=seed)
    return train, val, test

def scenario_level_split(df: pd.DataFrame):
    train_ids, val_ids, test_ids = [], [], []

    for label in sorted(df[TARGET_COL].unique()):
        scenarios = df[df[TARGET_COL] == label]["scenario_id"].unique()
        tr, va, te = split_scenario_ids(scenarios)
        train_ids.extend(tr); val_ids.extend(va); test_ids.extend(te)

    df_train = df[df["scenario_id"].isin(train_ids)].copy()
    df_val   = df[df["scenario_id"].isin(val_ids)].copy()
    df_test  = df[df["scenario_id"].isin(test_ids)].copy()

    for name, split in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
        print(f"{name:5s} → shape: {split.shape} | labels: {split[TARGET_COL].value_counts().to_dict()}")

    return df_train, df_val, df_test

df_train, df_val, df_test = scenario_level_split(df)

Train → shape: (23100, 131) | labels: {2: 7700, 1: 7700, 0: 7700}
Val   → shape: (4200, 131) | labels: {2: 1400, 1: 1400, 0: 1400}
Test  → shape: (4200, 131) | labels: {2: 1400, 1: 1400, 0: 1400}


- We are splittitng by scenario and not random rows because if we are to split randomly timesteps 1-699 of a scenario end up in train and 700 ends up in testing.
- The model basically would have een the scansion during training leading to data leakage.
- Scenario level split guarantees all 700 timesteps of a scenario stay in the same split.

In [5]:
# Save metadata
df_train[["scenario_id", "timestep"]].to_csv(OUTPUT_PATH / "train_meta.csv", index=False)
df_val[["scenario_id", "timestep"]].to_csv(OUTPUT_PATH / "val_meta.csv", index=False)
df_test[["scenario_id", "timestep"]].to_csv(OUTPUT_PATH / "test_meta.csv", index=False)

### Extracting features and targets

In [6]:
def extract_Xy(df_train, df_val, df_test, feature_cols):
    X_train = df_train[feature_cols].copy()
    X_val   = df_val[feature_cols].copy()
    X_test  = df_test[feature_cols].copy()

    y_train = df_train[TARGET_COL].copy()
    y_val   = df_val[TARGET_COL].copy()
    y_test  = df_test[TARGET_COL].copy()

    print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")
    return X_train, X_val, X_test, y_train, y_val, y_test

X_train, X_val, X_test, y_train, y_val, y_test = extract_Xy(df_train, df_val, df_test, feature_cols)

X_train: (23100, 126) | X_val: (4200, 126) | X_test: (4200, 126)


### Clean Features

In [7]:
def clean_features(X_train, X_val, X_test):
    print(f"Inf values in X_train : {np.isinf(X_train.select_dtypes(include=np.number)).sum().sum()}")
    print(f"NaN values in X_train : {X_train.isnull().sum().sum()}")

    cleaned = []
    for X in [X_train, X_val, X_test]:
        X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        X = X.clip(lower=X.quantile(0.001), upper=X.quantile(0.999), axis=1)
        cleaned.append(X)

    print("Clean-up done.\n")
    return cleaned[0], cleaned[1], cleaned[2]

X_train, X_val, X_test = clean_features(X_train, X_val, X_test)

Inf values in X_train : 0
NaN values in X_train : 0
Clean-up done.



### Scale and Save

In [8]:
def scale_and_save(scaler_name, scaler, X_train, X_val, X_test, feature_cols, save_y=False):
    # Fit on train only, transform all splits
    X_tr = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index)
    X_va = pd.DataFrame(scaler.transform(X_val),       columns=feature_cols, index=X_val.index)
    X_te = pd.DataFrame(scaler.transform(X_test),      columns=feature_cols, index=X_test.index)

    # Save scaled CSVs
    X_tr.to_csv(OUTPUT_PATH / f"X_train_{scaler_name}.csv", index=False)
    X_va.to_csv(OUTPUT_PATH / f"X_val_{scaler_name}.csv",   index=False)
    X_te.to_csv(OUTPUT_PATH / f"X_test_{scaler_name}.csv",  index=False)

    # Save fitted scaler — using your exact naming convention
    scaler_filename = f"robust_scaler.joblib" if scaler_name == "robust" else f"standard_scaler.joblib"
    joblib.dump(scaler, OUTPUT_PATH / scaler_filename)

    # Verify it loads back cleanly
    loaded = joblib.load(OUTPUT_PATH / scaler_filename)
    n_features = len(loaded.scale_) if hasattr(loaded, "scale_") else len(loaded.mean_)
    print(f"\n── {scaler_name.upper()} SCALER")
    print(f"Saved & verified — {n_features} features")
    print(f"Scaler path : {OUTPUT_PATH / scaler_filename}")
    print(f"X_train     : {X_tr.shape} | X_val: {X_va.shape} | X_test: {X_te.shape}")

    # Sanity check on a sample column
    s = X_tr["node_b_pressure"]
    print(f"node_b_pressure → mean={s.mean():.4f}  median={s.median():.4f}  std={s.std():.4f}  min={s.min():.4f}  max={s.max():.4f}")

    return X_tr, X_va, X_te

# Save y splits once (scaler-independent)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
y_train.to_csv(OUTPUT_PATH / "y_train.csv", index=False)
y_val.to_csv(  OUTPUT_PATH / "y_val.csv",   index=False)
y_test.to_csv( OUTPUT_PATH / "y_test.csv",  index=False)

# Run both scalers
scaled_results = {}
for scaler_name, scaler in SCALERS.items():
    X_tr_sc, X_va_sc, X_te_sc = scale_and_save(
        scaler_name, scaler, X_train, X_val, X_test, feature_cols
    )
    scaled_results[scaler_name] = (X_tr_sc, X_va_sc, X_te_sc)

print("\nPREPROCESSING COMPLETE — both scalers saved and ready.")


── ROBUST SCALER
Saved & verified — 126 features
Scaler path : /home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/model/scaled/robust_scaler.joblib
X_train     : (23100, 126) | X_val: (4200, 126) | X_test: (4200, 126)
node_b_pressure → mean=0.2206  median=-0.0000  std=3.4583  min=-12.9315  max=15.1090

── STANDARD SCALER
Saved & verified — 126 features
Scaler path : /home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/model/scaled/standard_scaler.joblib
X_train     : (23100, 126) | X_val: (4200, 126) | X_test: (4200, 126)
node_b_pressure → mean=-0.0000  median=-0.0638  std=1.0000  min=-3.8032  max=4.3052

PREPROCESSING COMPLETE — both scalers saved and ready.
